# E-Commerce Revenue Leakage & Profitability Analysis

##  Data Cleaning & Validation

### Objective
The purpose of this notebook is to prepare a clean and analysis-ready version of the synthetic e-commerce dataset.

The cleaning process focuses on:
- Correct data types
- Missing-value treatment
- Duplicate validation
- Categorical consistency
- Numerical range validation
- Business-rule validation
- Preserving legitimate loss-making transactions

No valid business records will be removed simply because they contain extreme losses or zero net revenue.

In [1]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
file_path = "../data/ecommerce_revenue_leakage_100k.csv"
df = pd.read_csv(file_path)
print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (100000, 31)


In [3]:
clean_df = df.copy()
print("Original shape:", df.shape)
print("Working copy shape:", clean_df.shape)

Original shape: (100000, 31)
Working copy shape: (100000, 31)


In [4]:
clean_df["order_date"] = pd.to_datetime(clean_df["order_date"],errors="coerce")
print("Order Date Data Type:", clean_df["order_date"].dtype)
print("Invalid Dates:", clean_df["order_date"].isna().sum())

Order Date Data Type: datetime64[us]
Invalid Dates: 0


## 4.2 Missing Value Treatment

Missing values are reviewed based on business context rather than being automatically dropped or imputed.

The `return_reason` field is only applicable to returned orders. Therefore, missing values for non-returned orders represent "Not Applicable" rather than unknown or poor-quality data.

In [5]:
missing_before = clean_df.isnull().sum()
missing_before[missing_before > 0]

return_reason    91867
dtype: int64

In [6]:
returned_missing_reason = clean_df.loc[clean_df["order_status"] == "Returned","return_reason"].isna().sum()
non_returned_missing_reason = clean_df.loc[clean_df["order_status"] != "Returned","return_reason"].isna().sum()
print("Returned orders missing return reason:", returned_missing_reason)
print("Non-returned orders missing return reason:", non_returned_missing_reason)

Returned orders missing return reason: 0
Non-returned orders missing return reason: 91867


In [7]:
clean_df["return_reason"] = clean_df["return_reason"].fillna("Not Applicable")

In [8]:
print("Remaining Missing Values:",clean_df.isnull().sum().sum())
print("\nReturn Reason Distribution:")
print(clean_df["return_reason"].value_counts())

Remaining Missing Values: 0

Return Reason Distribution:
return_reason
Not Applicable     91867
Size/Fit Issue      2169
Late Delivery       2028
Changed Mind        1007
Quality Issue       1001
Wrong Item           997
Damaged Product      931
Name: count, dtype: int64


## 4.3 Duplicate & Identifier Validation

This section verifies that the cleaned dataset contains no duplicate records and that key business identifiers are valid.

`order_id` is expected to uniquely identify each order, while customer and product identifiers may legitimately appear across multiple orders.

In [9]:
duplicate_rows = clean_df.duplicated().sum()
print("Duplicate Rows:", duplicate_rows)

Duplicate Rows: 0


In [10]:
duplicate_order_ids = clean_df["order_id"].duplicated().sum()
print("Duplicate Order IDs:", duplicate_order_ids)

Duplicate Order IDs: 0


In [11]:
print("Missing Order IDs:", clean_df["order_id"].isna().sum())
print("Missing Customer IDs:", clean_df["customer_id"].isna().sum())
print("Missing Product IDs:", clean_df["product_id"].isna().sum())

Missing Order IDs: 0
Missing Customer IDs: 0
Missing Product IDs: 0


In [12]:
id_columns = ["order_id","customer_id","product_id"]
for column in id_columns:
    blank_count = (
        clean_df[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    print(f"Blank {column}: {blank_count}")

Blank order_id: 0
Blank customer_id: 0
Blank product_id: 0


In [13]:
print("Total Rows:", len(clean_df))
print("Unique Order IDs:", clean_df["order_id"].nunique())
print("Unique Customer IDs:", clean_df["customer_id"].nunique())
print("Unique Product IDs:", clean_df["product_id"].nunique())

Total Rows: 100000
Unique Order IDs: 100000
Unique Customer IDs: 19872
Unique Product IDs: 2500


## 4.4 Categorical Value Validation

This section validates categorical fields for unexpected values, inconsistent labels, and formatting issues before further analysis.

In [14]:
categorical_columns = clean_df.select_dtypes(include=["object", "string"]).columns.tolist()
print("Categorical Columns:")
print(categorical_columns)
print("\nTotal Categorical Columns:", len(categorical_columns))

Categorical Columns:
['order_id', 'customer_id', 'customer_segment', 'product_id', 'product_name', 'category', 'sub_category', 'region', 'state', 'payment_method', 'order_status', 'return_reason', 'leakage_reason']

Total Categorical Columns: 13


In [15]:
for column in categorical_columns:
    print(f"{column}: {clean_df[column].nunique()} unique values")

order_id: 100000 unique values
customer_id: 19872 unique values
customer_segment: 3 unique values
product_id: 2500 unique values
product_name: 3600 unique values
category: 4 unique values
sub_category: 12 unique values
region: 4 unique values
state: 14 unique values
payment_method: 5 unique values
order_status: 3 unique values
return_reason: 7 unique values
leakage_reason: 6 unique values


In [16]:
business_categories = [
    "customer_segment",
    "category",
    "sub_category",
    "region",
    "state",
    "payment_method",
    "order_status",
    "return_reason",
    "leakage_reason"
]
for column in business_categories:
    print(f"\n--- {column.upper()} ---")
    print(sorted(clean_df[column].astype(str).unique()))


--- CUSTOMER_SEGMENT ---
['Consumer', 'Corporate', 'Home Office']

--- CATEGORY ---
['Beauty', 'Electronics', 'Fashion', 'Home & Furniture']

--- SUB_CATEGORY ---
['Accessories', 'Footwear', 'Fragrances', 'Furniture', 'Hair Care', 'Home Decor', 'Kitchen', 'Laptops', 'Men Clothing', 'Skin Care', 'Smartphones', 'Women Clothing']

--- REGION ---
['East', 'North', 'South', 'West']

--- STATE ---
['Bihar', 'Delhi', 'Goa', 'Gujarat', 'Karnataka', 'Kerala', 'Maharashtra', 'Odisha', 'Punjab', 'Rajasthan', 'Tamil Nadu', 'Telangana', 'Uttar Pradesh', 'West Bengal']

--- PAYMENT_METHOD ---
['COD', 'Credit Card', 'Debit Card', 'Net Banking', 'UPI']

--- ORDER_STATUS ---
['Cancelled', 'Completed', 'Returned']

--- RETURN_REASON ---
['Changed Mind', 'Damaged Product', 'Late Delivery', 'Not Applicable', 'Quality Issue', 'Size/Fit Issue', 'Wrong Item']

--- LEAKAGE_REASON ---
['Cancellation Loss', 'Excessive Discount', 'Healthy Order', 'High Shipping Cost', 'Low-Margin Product', 'Return Loss']


In [17]:
for column in categorical_columns:
    whitespace_count = (
        clean_df[column].astype(str)
        != clean_df[column].astype(str).str.strip()
    ).sum()
    if whitespace_count > 0:
        print(f"{column}: {whitespace_count} values contain extra whitespace")

In [18]:
print("Order Status:")
print(clean_df["order_status"].value_counts())
print("\nReturned Flag:")
print(clean_df["returned"].value_counts())
print("\nLeakage Reason:")
print(clean_df["leakage_reason"].value_counts())

Order Status:
order_status
Completed    88644
Returned      8133
Cancelled     3223
Name: count, dtype: int64

Returned Flag:
returned
0    91867
1     8133
Name: count, dtype: int64

Leakage Reason:
leakage_reason
Healthy Order         57140
Low-Margin Product    15187
High Shipping Cost    10130
Return Loss            8133
Excessive Discount     6187
Cancellation Loss      3223
Name: count, dtype: int64


## 4.5 Numerical Range & Business Rule Validation

This section validates numerical fields and core business rules to ensure that the dataset is logically consistent.

Negative profit values are not automatically treated as data errors because loss-making orders are important to the revenue leakage analysis.

In [19]:
validation_checks = {
    "Quantity <= 0": (clean_df["quantity"] <= 0).sum(),
    "Unit Price <= 0": (clean_df["unit_price"] <= 0).sum(),
    "Gross Revenue <= 0": (clean_df["gross_revenue"] <= 0).sum(),
    "Discount % < 0": (clean_df["discount_pct"] < 0).sum(),
    "Discount % > 100": (clean_df["discount_pct"] > 100).sum(),
    "Discount Amount < 0": (clean_df["discount_amount"] < 0).sum(),
    "COGS < 0": (clean_df["cogs"] < 0).sum(),
    "Shipping Cost < 0": (clean_df["shipping_cost"] < 0).sum(),
    "Delivery Days < 0": (clean_df["delivery_days"] < 0).sum(),
    "Return Loss < 0": (clean_df["return_loss"] < 0).sum(),
    "Cancellation Loss < 0": (clean_df["cancellation_loss"] < 0).sum(),
    "Revenue Leakage < 0": (clean_df["revenue_leakage"] < 0).sum()
}

for check, count in validation_checks.items():
    print(f"{check}: {count}")

Quantity <= 0: 0
Unit Price <= 0: 0
Gross Revenue <= 0: 0
Discount % < 0: 0
Discount % > 100: 0
Discount Amount < 0: 0
COGS < 0: 0
Shipping Cost < 0: 0
Delivery Days < 0: 0
Return Loss < 0: 0
Cancellation Loss < 0: 0
Revenue Leakage < 0: 0


In [20]:
gross_revenue_valid = np.isclose(
    clean_df["gross_revenue"],
    clean_df["quantity"] * clean_df["unit_price"],
    atol=0.02
)
discount_valid = np.isclose(
    clean_df["discount_amount"],
    clean_df["gross_revenue"] * clean_df["discount_pct"] / 100,
    atol=0.02
)
sales_valid = np.isclose(
    clean_df["sales"],
    clean_df["gross_revenue"] - clean_df["discount_amount"],
    atol=0.02
)
print("Gross Revenue Mismatches:", (~gross_revenue_valid).sum())
print("Discount Amount Mismatches:", (~discount_valid).sum())
print("Sales Formula Mismatches:", (~sales_valid).sum())

Gross Revenue Mismatches: 0
Discount Amount Mismatches: 0
Sales Formula Mismatches: 0


In [21]:
returned_flag_mismatch = (
    ((clean_df["order_status"] == "Returned") & (clean_df["returned"] != 1)) |
    ((clean_df["order_status"] != "Returned") & (clean_df["returned"] != 0))
).sum()
return_loss_mismatch = (
    ((clean_df["order_status"] == "Returned") & (clean_df["return_loss"] <= 0)) |
    ((clean_df["order_status"] != "Returned") & (clean_df["return_loss"] > 0))
).sum()
cancellation_loss_mismatch = (
    ((clean_df["order_status"] == "Cancelled") & (clean_df["cancellation_loss"] <= 0)) |
    ((clean_df["order_status"] != "Cancelled") & (clean_df["cancellation_loss"] > 0))
).sum()
print("Returned Flag Mismatches:", returned_flag_mismatch)
print("Return Loss Mismatches:", return_loss_mismatch)
print("Cancellation Loss Mismatches:", cancellation_loss_mismatch)

Returned Flag Mismatches: 0
Return Loss Mismatches: 0
Cancellation Loss Mismatches: 0


## 4.6 Final Data Quality Check & Export

After completing the cleaning and validation process, the final dataset is checked for missing values, duplicates, invalid identifiers, and structural consistency.

The cleaned dataset will be exported as a separate CSV file so that the original synthetic dataset remains unchanged.

In [22]:
print("Final Dataset Shape:", clean_df.shape)
print("\nData Quality Summary")
print("--------------------")
print("Total Rows:", len(clean_df))
print("Total Columns:", clean_df.shape[1])
print("Missing Values:", clean_df.isnull().sum().sum())
print("Duplicate Rows:", clean_df.duplicated().sum())
print("Duplicate Order IDs:", clean_df["order_id"].duplicated().sum())
print("\nDate Range")
print("Start Date:", clean_df["order_date"].min())
print("End Date:", clean_df["order_date"].max())

Final Dataset Shape: (100000, 31)

Data Quality Summary
--------------------
Total Rows: 100000
Total Columns: 31
Missing Values: 0
Duplicate Rows: 0
Duplicate Order IDs: 0

Date Range
Start Date: 2024-01-01 00:00:00
End Date: 2025-12-31 00:00:00


In [23]:
original_rows = len(df)
cleaned_rows = len(clean_df)
print("Original Rows:", original_rows)
print("Cleaned Rows:", cleaned_rows)
print("Rows Removed:", original_rows - cleaned_rows)

Original Rows: 100000
Cleaned Rows: 100000
Rows Removed: 0


In [24]:
output_path = "../data/ecommerce_revenue_leakage_cleaned.csv"
clean_df.to_csv(output_path,index=False)
print("Cleaned dataset saved successfully.")
print("File:", output_path)

Cleaned dataset saved successfully.
File: ../data/ecommerce_revenue_leakage_cleaned.csv


In [ ]:
test_df = pd.read_csv(output_path)
print("Exported Dataset Shape:", test_df.shape)
print("Exported Missing Values:", test_df.isnull().sum().sum())
print("Exported Duplicate Rows:", test_df.duplicated().sum())

Exported Dataset Shape: (100000, 31)
Exported Missing Values: 0
Exported Duplicate Rows: 0


: 